# Linearization experiments

Examples recovered from commented-out code in `visualize_linearization.ipynb`:
alternative spaCy models, cleft/passive example sentences and draft passive regexes.


In [ ]:
# Required models (uncomment to download):
# !python -m spacy download en_core_web_lg
# !python -m spacy download ru_core_news_lg
# !python -m spacy download zh_core_web_sm


In [ ]:
import re

import spacy
from spacy import displacy

from gram2vec import SyntaxRegexMatcher

DISPLACY_OPTIONS = {
    'compact': True,
    'bg': '#ffffff',
    'color': '#000000',
    'font': 'Source Sans Pro',
}


def show(nlp, matcher, text, render=True):
    """Render the dependency parse, print linearized trees and pattern matches."""
    doc = nlp(text)
    if render:
        displacy.render(doc, style='dep', options=DISPLACY_OPTIONS)
    for sent in doc.sents:
        print(matcher.linearize_tree(sent))
    for match in matcher.match_document(doc):
        print('MATCH', match)
    return doc


## English


In [ ]:
nlp_en = spacy.load('en_core_web_lg')
matcher_en = SyntaxRegexMatcher(language='en')


In [ ]:
show(nlp_en, matcher_en, 'It was John who bought the butter.')  # it-cleft


In [ ]:
# Expected (partial) linearization:
# (was-be-VBD-ROOT(It-it-PRP-nsubj)(car-car-NN-attr(Jane-Jane-NNP-poss(’s-’s-POS-case)(stolen-steal-VBN-relcl
show(nlp_en, matcher_en, 'It was Jane’s car that got stolen last night.')


In [ ]:
show(nlp_en, matcher_en, 'Yesterday the book was bought.')  # passive


### Draft English passive regex


In [ ]:
passive_en = r'\([^-]*-[^-]*-(NN[^-]*|PRP|WDT)-nsubjpass.*\([^-]*-be-[^-]*-auxpass'

for text in ['Yesterday the book was bought.', 'It was Jane’s car that got stolen last night.']:
    for sent in nlp_en(text).sents:
        tree = matcher_en.linearize_tree(sent)
        print(bool(re.search(passive_en, tree)), '|', sent.text)


## Russian


In [ ]:
nlp_ru = spacy.load('ru_core_news_lg')
matcher_ru = SyntaxRegexMatcher(language='ru')


In [ ]:
show(nlp_ru, matcher_ru, 'Это Джон купил масло.')  # cleft-like construction


In [ ]:
show(nlp_ru, matcher_ru, 'Вчера книга была куплена и прочитана.')  # analytic passive


### Draft Russian analytic passive regex

Passive subject plus a `быть` auxiliary (`aux:pass`). The original draft lacked a group around
the auxiliary alternation; it is fixed below.


In [ ]:
passive_aux_ru = (
    r'\([^-]*-[^-]*-(NOUN[^-]*|PRON)-nsubj:pass.*'
    r'\((будет|будут|была|был|были)-[^-]*-[^-]*-aux:pass'
)

for text in ['Вчера книга была куплена и прочитана.', 'Это Джон купил масло.']:
    for sent in nlp_ru(text).sents:
        tree = matcher_ru.linearize_tree(sent)
        print(bool(re.search(passive_aux_ru, tree)), '|', sent.text)


In [ ]:
# Register it alongside the built-in Russian patterns
matcher_ru.add_patterns({'passive_aux_rus': passive_aux_ru})
show(nlp_ru, matcher_ru, 'Вчера книга была куплена и прочитана.', render=False)


## Chinese (linearization only, no built-in patterns)


In [ ]:
nlp_zh = spacy.load('zh_core_web_sm')
matcher_zh = SyntaxRegexMatcher(language='zh')  # empty pattern set
show(nlp_zh, matcher_zh, '昨天这本书被买走了。')
